In [1]:
from nltk import word_tokenize
from nltk.corpus import stopwords
from collections import Counter
import math
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\franc\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\franc\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\franc\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [2]:
corpus = [
    "the sky is blue",
    "the sun is bright",
    "the sun in the sky"
]


In [3]:
# Tokenize text
def tokenizer(doc): 
    return [w.lower() for w in word_tokenize(doc) if w.lower() not in stopwords.words('english')]
corpus_tokens = [tokenizer(doc) for doc in corpus]

print(corpus_tokens)

[['sky', 'blue'], ['sun', 'bright'], ['sun', 'sky']]


In [4]:
# Compute term frequency for one term in one document
def tf(t, doc_tokens):
    N = len(doc_tokens)
    res = 0
    for t_ in doc_tokens:
        if t == t_:
            res += 1
    return res / N

# Compute TF values for all terms in a document
def doc_tf_old(doc_tokens):
    res = {}
    N = len(doc_tokens)
    for t in doc_tokens:
        res[t] = tf(t,doc_tokens) 
    return res


def doc_tf(doc_tokens):
    N = len(doc_tokens)
    count = Counter(doc_tokens)
    for c in count:
        count[c] /= N 
    return count

# Compute IDF values for all terms in the corpus
def idf_aux(t, corpus_tokens):
    N = len(corpus_tokens)
    df = 0
    for d in corpus_tokens:
        if t in d:
            df = df + 1
    return math.log( N / df, 10)
    
# Compute IDF values for all terms in the corpus
def idf(corpus_tokens):
    tokens = set([token for d in corpus_tokens for token in d])
    res = {}
    for t in tokens:
        res[t] = idf_aux(t,corpus_tokens)
    return res

#print(idf(corpus_tokens))

# Compute TF-IDF representation for each document
def tf_idf(corpus_tokens):
    res = []
    idf_dict = idf(corpus_tokens)
    for doc_tokens in corpus_tokens:
        tf_idf_dict = {}
        tf_dict = doc_tf(doc_tokens)
        for termo in tf_dict:
            tf_idf_dict[termo] = idf_dict[termo] * tf_dict[termo]
        res.append(tf_idf_dict)
    return res  


In [5]:
# Convert TF-IDF dictionaries into numerical vectors
def vectorize(tf_idf_list):
    vocab = set([token for d in corpus_tokens for token in d])
    res = []
    for doc in tf_idf_list:
        res.append([doc.get(token, 0) for token in vocab])
    return res

In [6]:
# Build document vectors
tf_idf_list = tf_idf(corpus_tokens)
doc_vectors = vectorize(tf_idf_list)
print(doc_vectors)

query = "The bright sun"

query_tokens = tokenizer(query)

[[0, 0, 0.08804562952784062, 0.23856062735983122], [0.08804562952784062, 0.23856062735983122, 0, 0], [0.08804562952784062, 0, 0.08804562952784062, 0]]


In [7]:
# Convert the query into a TF-IDF vector
def query_vector(query_tokens, idf_dict, vocab):
    # Compute TF for the query
    tf_dict = doc_tf(query_tokens)

    vector = []

    # Build the query vector using the same vocabulary as the documents
    for term in vocab:
        tf_value = tf_dict.get(term, 0)
        idf_value = idf_dict.get(term, 0)

        vector.append(tf_value * idf_value)

    return vector

In [8]:
# Create the vocabulary used by the document vectors
vocab = list(set([token for doc in corpus_tokens for token in doc]))

idf_dict = idf(corpus_tokens)
q_vector = query_vector(query_tokens, idf_dict, vocab)

print("Vocabulary:", vocab)
print("Query vector:", q_vector)

Vocabulary: ['sun', 'bright', 'sky', 'blue']
Query vector: [0.08804562952784062, 0.23856062735983122, 0.0, 0.0]


In [9]:
# Compute cosine similarity between two vectors
def cosine_similarity(v1, v2):
    dot_product = sum(a * b for a, b in zip(v1, v2))

    norm_v1 = math.sqrt(sum(a * a for a in v1))
    norm_v2 = math.sqrt(sum(b * b for b in v2))

    if norm_v1 == 0 or norm_v2 == 0:
        return 0

    return dot_product / (norm_v1 * norm_v2)

In [10]:
# Compare the query with each document
similarities = []

for i, doc_vector in enumerate(doc_vectors):
    score = cosine_similarity(q_vector, doc_vector)
    similarities.append((i + 1, score))

# Sort documents from most relevant to least relevant
ranking = sorted(similarities, key=lambda x: x[1], reverse=True)

print("Final ranking:")

for doc_id, score in ranking:
    print(f"D{doc_id}: {corpus[doc_id - 1]} --> Similarity: {score:.4f}")

Final ranking:
D2: the sun is bright --> Similarity: 1.0000
D3: the sun in the sky --> Similarity: 0.2448
D1: the sky is blue --> Similarity: 0.0000
